# Bat Species Occurrence Records with Taxonomic, Geographic, and Environmental Metadata from Sub-Saharan Africa Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the FAIR² bat occurrence dataset using the `mlcroissant` library.

### Dataset Source

Dataset source is defined by a Croissant schema:

[https://sen.science/doi/10.71728/senscience.y2kq-ta3n/fair2.json](https://sen.science/doi/10.71728/senscience.y2kq-ta3n/fair2.json)


In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y2kq-ta3n/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

# Show keywords and temporal coverage
print("Keywords:", metadata.keywords)
print("Temporal coverage:", metadata.temporalCoverage)
print("Spatial coverage:", metadata.spatialCoverage)

## 2. Data Overview

Review available record sets and fields. All entities are referenced by their `@id`.

We'll list each RecordSet's `@id` and the `@id` for each Field within.

In [ ]:
# List all recordSets by @id
record_sets = [rs['@id'] if isinstance(rs, dict) else rs for rs in getattr(metadata, 'recordSet', [])]
if not record_sets:
    print("No recordSets found in metadata.")

# For demonstration, reload metadata from the Croissant JSON to get recordSets (because the recordSet field may be empty in mlcroissant's dataset object).
import requests

response = requests.get(croissant_url)
raw_metadata = response.json()

def extract_record_sets(jsonld):
    return [x for x in jsonld.get('recordSet', [])] if 'recordSet' in jsonld else []

record_sets = extract_record_sets(raw_metadata)
record_set_ids = []
for rs in record_sets:
    if isinstance(rs, dict) and '@id' in rs:
        record_set_ids.append(rs['@id'])
    elif isinstance(rs, str):
        record_set_ids.append(rs)

# Show recordSet IDs
print("RecordSet @id values:")
for rid in record_set_ids:
    print(" -", rid)

## For each RecordSet, fetch its Fields and their @id
print("\nFields by RecordSet:")
for rid in record_set_ids:
    # Find the recordSet object
    rs_obj = None
    for item in raw_metadata.get('recordSet', []):
        if (isinstance(item, dict) and item.get('@id') == rid):
            rs_obj = item
            break
    if rs_obj:
        fields = rs_obj.get('field', [])
        field_ids = [f['@id'] if isinstance(f, dict) and '@id' in f else f for f in fields]
        print(f"RecordSet {rid}:")
        for fid in field_ids:
            print(f"    Field @id: {fid}")
    else:
        print(f"RecordSet {rid}: No fields found or not accessible.")

## 3. Data Extraction

Load data from each record set into a DataFrame for analysis using its `@id`.

*Note*: Depending on the actual content of the Croissant schema, the recordSet list may need to be updated. For the FAIR² bat dataset, let's assume the main occurrence records are within a record set like:

`https://api.app.sen.science/frontiers/7988377/occurrence-records` (You may need to update this with the actual value from the overview above!)


In [ ]:
# Choose main record set @id(s) from above
# Example: Main record set may be 'https://api.app.sen.science/frontiers/7988377/occurrence-records'
# Replace with actual @id(s) obtained from overview
record_sets_to_extract = record_set_ids  # Use all available
dataframes = {}

for record_set_id in record_sets_to_extract:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            dataframes[record_set_id] = pd.DataFrame(records)
            print(f"Loaded {len(records)} records for record set {record_set_id}")
    except Exception as e:
        print(f"Could not load records for {record_set_id}: {e}")

# Show columns for one record set
if dataframes:
    example_record_set = list(dataframes.keys())[0]
    print(f"Fields (@id) in columns for {example_record_set}:")
    print(dataframes[example_record_set].columns.tolist())
    display(dataframes[example_record_set].head())
else:
    print("No DataFrames loaded. Please check recordSet @id values.")

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps: filtering, normalization, grouping.

You must reference fields and columns by their `@id`. The following is a generic template; replace `<numeric_field_id>` and `<group_field_id>` with actual values from data overview.

In [ ]:
# Example EDA using actual field @id
if dataframes:
    df = dataframes[example_record_set]
    # Find a numeric column by @id (e.g. Area of Occupancy, Extent of Occurrence)
    # Placeholder: Replace with actual field @id from above
    potential_numeric_fields = [col for col in df.columns if df[col].dtype in ['int64', 'float64']]
    if potential_numeric_fields:
        numeric_field = potential_numeric_fields[0]
        print(f"Using numeric field: {numeric_field}")
        threshold = df[numeric_field].mean() # Example threshold: mean value
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())

        # Normalize numeric field
        filtered_df[numeric_field + '_normalized'] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, numeric_field + '_normalized']].head())

        # Group by a categorical field by @id (e.g. species @id or country @id)
        potential_group_fields = [col for col in df.columns if df[col].dtype == 'object']
        if potential_group_fields:
            group_field = potential_group_fields[0]
            print(f"Grouping by field: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index(name=f"mean_{numeric_field}")
            print(f"Grouped data by {group_field}:")
            display(grouped_df.head())
        else:
            print("No suitable group fields found.")
    else:
        print("No numeric fields available for EDA.")
else:
    print("No DataFrames available for EDA.")

## 5. Visualization

Visualize the distribution of the numeric field and relation with the group field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and potential_numeric_fields and potential_group_fields:
    plt.figure(figsize=(10, 6))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # Plot mean numeric field per group
    if 'grouped_df' in locals():
        plt.figure(figsize=(12, 6))
        sns.barplot(x=group_field, y=f"mean_{numeric_field}", data=grouped_df.head(20))
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion

This notebook shows how to use the `mlcroissant` library to load, extract, and analyze a FAIR² dataset using unique `@id` references for all entities.

Key findings:
- The dataset contains rich occurrence records for African bat species, with spatial, temporal, and taxonomic metadata.
- Numeric fields such as conservation metrics can be analyzed and visualized per group (e.g., species or country).

You can further explore the dataset using its Croissant schema, and reference record sets, fields, and columns by `@id` for reproducible, FAIR analysis.